# 3B @ FP16 — pre-flight smoke test

**Purpose: prove the ONE path that could not be tested locally.**

Before the final 24-hour sweep, three of four code paths are already verified on a
4 GB card: 3B@4-bit (full run), 3B@8-bit (full run, with the nested-array tic
reproduced live and recovered), and the OOM autotune (forced with a capped
allocator; halves correctly, no records lost or duplicated).

**3B @ fp16 needs 6.2 GB and could not be run locally at all.** It is also the
reference cell that all 21 runs are measured against, so if it is broken every
number in the sweep is worthless. This notebook runs it on 20 questions first.

## What we are looking for

The 3B family has a known failure mode absent at 1.5B — `{"spans": [[...]]}`,
nested one level too deep. Measured at n=3000: **330 occurrences at fp16, 600 at
8-bit, 13 at 4-bit, 0 at 1.5B.** `parsing.salvage()` now unwraps it without
touching `parse_status`. Cell 5 checks that it fires and recovers correctly.

The other failure mode — unescaped inner double quotes from copying `"Tunnels &
Trolls"` verbatim into a JSON string — is **deliberately not fixed**. Expect a few
percent of `malformed_json` on the extractor at every model size. That is a
task/format conflict, not a defect.

## Settings (Settings panel, right)

| | |
|---|---|
| Accelerator | **GPU** (A100 if offered, else T4 — 3B/fp16 needs >6.2 GB) |
| Internet | **On** |
| Persistence | not required for a smoke test |


In [ ]:
!pip install -q -U "transformers==5.14.1" "bitsandbytes==0.50.0" "datasets==5.0.1" accelerate pyyaml


In [ ]:
import os
os.environ['HF_HOME'] = '/kaggle/tmp/hf'
os.makedirs('/kaggle/tmp/hf', exist_ok=True)

REPO_URL = 'https://github.com/Retixx/Maxim-Mohareb-Michael-Zhang-Fun-Time.git'
REPO_DIR = '/kaggle/working/marag-precision'
BRANCH   = 'final-3b-reference'   # the final run config

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q --ff-only
else:
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

import torch
free, total = torch.cuda.mem_get_info()
print(f'{torch.cuda.get_device_name(0)}  free {free/1024**3:.1f} GB of {total/1024**3:.1f} GB')
assert torch.cuda.is_available(), 'No GPU - set Accelerator in Settings'
# 3B fp16 weights are ~6.2 GB; extractor prompts reach ~3.7k tokens and the KV
# cache at batch 32 is the real cost. Refuse to start on a card that cannot hold it.
assert free/1024**3 > 12, f'need >12 GB free for 3B fp16 at batch 32, have {free/1024**3:.1f}'
print('OK to proceed')


### The smoke test

20 questions on the **development seed (1234)**, never seed 7 — the sweep's
evaluation questions must stay unseen (`src/prompts.py` seed-hygiene note).

`baseline` = all four agents at Qwen2.5-3B @ fp16, i.e. the reference cell.
Batch 32, matching what the sweep will pin.


In [ ]:
!python -X utf8 -m src.runner --config config/experiment.yaml --run baseline --n 20 --seed 1234 --batch-size 32


### Verdict

Reads the JSONL the run just wrote and decides GO / NO-GO against explicit
thresholds. Do not eyeball this — the whole point is a decision rule fixed before
the data exists.


In [ ]:
import json, collections, glob
from src.parsing import salvage

f = sorted(glob.glob('results/baseline_qwen2.5-3b_n20_seed1234.jsonl'))[0]
recs = [json.loads(l) for l in open(f, encoding='utf-8')]
calls = [x for x in recs if x.get('record_type') != 'answer']
ans   = [x for x in recs if x.get('record_type') == 'answer']
meta  = json.load(open(f.replace('.jsonl', '.meta.json'), encoding='utf-8'))

by = collections.defaultdict(collections.Counter)
for x in calls:
    by[x['stage']][x['parse_status']] += 1

print('parse status per stage')
for s in ['planner', 'step_definer', 'extractor', 'qa']:
    t = sum(by[s].values())
    ok = 100 * by[s]['ok'] / t if t else 0
    rest = {k: v for k, v in by[s].items() if k != 'ok'}
    print(f'  {s:<14} {t:>4} calls  ok {ok:5.1f}%   {rest or "-"}')

em = 100 * sum(a['em'] for a in ans) / len(ans)
f1 = 100 * sum(a['f1'] for a in ans) / len(ans)
ex_ok = 100 * by['extractor']['ok'] / max(sum(by['extractor'].values()), 1)
st = meta['stages']
bs = {k: v['final_batch_size'] for k, v in st.items()}
peak = max(v['peak_vram_mb'] for v in st.values())

print(f'\nEM {em:.1f}%   F1 {f1:.1f}%   n={len(ans)}')
print(f'batch per stage {bs}   peak VRAM {peak:.0f} MB')
print(f'model_id on every record: {sorted({x.get("model_id","MISSING").split("/")[-1] for x in calls})}')

# the 3B-specific tic, and whether the fix catches it
nested = [x for x in calls if x['stage'] == 'extractor'
          and x['raw_output'].strip().replace(' ', '').startswith('{"spans":[[')]
print(f'\nnested-array tic: {len(nested)} of {sum(by["extractor"].values())} extractor calls')
for x in nested[:2]:
    print(f'   [{x["parse_status"]}] {x["raw_output"][:90]!r}')
    print(f'   salvaged -> {salvage("extractor", x["raw_output"])}')
salv = sum(1 for x in calls if x['stage'] == 'extractor'
           and x['parse_status'] != 'ok' and salvage('extractor', x['raw_output']))
print(f'extractor failures recovered by salvage: {salv}')

print('\n' + '=' * 62)
checks = [
    ('ran end to end (20 answers)',        len(ans) == 20),
    ('batch stayed at 32 (no autotune)',   set(bs.values()) == {32}),
    ('metadata complete',                  meta.get('metadata_complete') is True),
    ('footprint recorded',                 meta.get('deduped_footprint_mb') is not None),
    ('planner parse > 90%',                100*by['planner']['ok']/max(sum(by['planner'].values()),1) > 90),
    ('extractor parse > 80%',              ex_ok > 80),
    ('EM in 20-60% (sane at n=20)',        20 <= em <= 60),
]
for name, passed in checks:
    print(f'  {"PASS" if passed else "FAIL"}  {name}')
print('=' * 62)
print('GO - launch the sweep' if all(p for _, p in checks)
      else 'NO-GO - do not spend the 24h window; send this output back')
